# ShopGenie
Redifining the concept of Online shopping customer experience with agentic AI.

## Key Features:
- **Tavily** for web search.
- **llama-3.1-70B** for arranging the data into specific schema and comparing products.
Tells the best product among the searched ones.
- **Youtube API** for providing the review link of the best product for self-satisfaction.
- **SMTP** for sending mail about the best product and its review to the user.

## Important packages
following are the required packeages for this agent to function.


```bash
pip install langchain-groq langgraph tavily-python google-api-python-client langchain-community beautifulsoup4
```

## Import necessary modules

In [3]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient
from langchain_community.tools import TavilySearchResults
from typing import List, Optional, Dict
from typing_extensions import TypedDict
from googleapiclient.discovery import build
from dotenv import load_dotenv
from IPython.display import Image, display
import getpass
import os
import json
import bs4
from langchain_community.document_loaders import WebBaseLoader
from pydantic import BaseModel, HttpUrl, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
import time

load_dotenv()

True

## Setting environment variables
These are the totally open source technologies used and environment variables which can be changed according to one's needs and availability

In [5]:
groq_api_key = os.getenv('GROQ_API_KEY')
tavily_api_key = os.getenv('TAVILY_API_KEY')
youtube_api_key = os.getenv('YOUTUBE_API_KEY')

#LLM being used in this notebook
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    api_key=groq_api_key,
    temperature=0.5,
)

#Tavily for web search
tavily_client = TavilyClient(api_key=tavily_api_key)

#Youtube api for video search
youtube = build('youtube', 'v3', developerKey=youtube_api_key)

In [6]:
class SpecsComparison(BaseModel):
    processor: str = Field(..., description="Processor type and model, e.g., 'Snapdragon 888'")
    battery: str = Field(..., description="Battery capacity and type, e.g., '4500mAh'")
    camera: str = Field(..., description="Camera specs, e.g., '108MP primary'")
    display: str = Field(..., description="Display type, size, refresh rate, e.g., '6.5 inch OLED, 120Hz'")
    storage: str = Field(..., description="Storage options and expandability, e.g., '128GB, expandable'")

class RatingsComparison(BaseModel):
    overall_rating: float = Field(..., description="Overall rating out of 5, e.g., 4.5")
    performance: float = Field(..., description="Rating for performance out of 5, e.g., 4.7")
    battery_life: float = Field(..., description="Rating for battery life out of 5, e.g., 4.3")
    camera_quality: float = Field(..., description="Rating for camera quality out of 5, e.g., 4.6")
    display_quality: float = Field(..., description="Rating for display quality out of 5, e.g., 4.8")

class Comparison(BaseModel):
    product_name: str = Field(..., description="Name of the product")
    specs_comparison: SpecsComparison
    ratings_comparison: RatingsComparison
    reviews_summary: str = Field(..., description="Summary of key points from user reviews about this product")

class BestProduct(BaseModel):
    product_name: str = Field(..., description="Name of the best product")
    justification: str = Field(..., description="Explanation of why this product is the best choice")

class ProductComparison(BaseModel):
    comparisons: List[Comparison]
    best_product: BestProduct

class Highlights(BaseModel):
    Camera: Optional[str] = None
    Performance: Optional[str] = None
    Display: Optional[str] = None
    Fast_Charging: Optional[str] = None

class SmartphoneReview(BaseModel):
    """A review of a smartphone."""
    title: str = Field(..., description="The title of the smartphone review")
    url: Optional[str] = Field(None, description="The URL of the smartphone review")
    content: Optional[str] = Field(None, description="The main content of the smartphone review")
    pros: Optional[List[str]] = Field(None, description="The pros of the smartphone")
    cons: Optional[List[str]] = Field(None, description="The cons of the smartphone")
    highlights: Optional[dict] = Field(None, description="The highlights of the smartphone")
    score: Optional[float] = Field(None, description="The score of the smartphone")

class ListOfSmartphoneReviews(BaseModel):
    """A list of smartphone reviews."""
    reviews: List[SmartphoneReview] = Field(..., description="List of individual smartphone reviews")

class EmailRecommendation(BaseModel):
    subject: str = Field(..., description="The email subject line, designed to capture the recipient's attention.")
    heading: str = Field(..., description="The main heading of the email, introducing the recommended product.")
    justification_line: str = Field(..., description="A concise explanation of why the product is being recommended.")

## Main State

In [7]:
class State(TypedDict):
    query: str
    email: str
    products: list[dict]
    product_schema: list[SmartphoneReview]
    blogs_content: Optional[list[dict]]
    best_product: dict
    comparsion: list
    youtube_link: str

## Sending Email
This is a complete function of sending mail which takes ShopGenie to next level and speaks of its potential

In [9]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv

load_dotenv()

def send_email(recipient_email, subject, body):
    """Send an email dynamically using SMTP."""
    # SMTP server configuration
    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    # Email credentials
    sender_email = os.getenv("GMAIL_USER")
    sender_password = os.getenv("GMAIL_PASS")

    try:
        # Create email content
        message = MIMEMultipart()
        message['From'] = sender_email
        message['To'] = recipient_email
        message['Subject'] = subject

        # Add the email body
        message.attach(MIMEText(body, 'html'))

        # Connect to the SMTP server
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()  # Start TLS encryption
            server.login(sender_email, sender_password)  # Login to the server
            server.send_message(message)  # Send the email
            print(f"Email sent successfully to {recipient_email}.")

    except Exception as e:
        print(f"Failed to send email: {e}")

In [10]:
#Email prompt template
email_template_prompt = """
You are an expert email content writer.

Generate an email recommendation based on the following inputs:
- Product Name: {product_name}
- Justification Line: {justification_line}
- User Query: "{user_query}" (a general idea of the user's interest, such as "a smartphone for photography" or "a premium gaming laptop").

Return your output in the following JSON format:
{format_instructions}

### Input Example:
Product Name: Google Pixel 8 Pro
Justification Line: Praised for its exceptional camera, advanced AI capabilities, and vibrant display.
User Query: a phone with an amazing camera

### Example Output:
{{
  "subject": "Capture Every Moment with Google Pixel 8 Pro",
  "heading": "Discover the Power of the Ultimate Photography Smartphone",
  "justification_line": "Known for its exceptional camera quality, cutting-edge AI features, and vibrant display, the Google Pixel 8 Pro is perfect for photography enthusiasts."
}}

Now generate the email recommendation based on the inputs provided.
"""

In [11]:

#email html template
email_html_template = """
    
    
        
    
    
        
            
                {heading}
            
            
                Our Top Pick: {product_name}
                {justification}
                Watch our in-depth review to explore why this phone is the best choice for you:
                Watch the Review
            
            
                
                    Want to learn more? Visit our website or follow us for more recommendations.
                    Explore Now
                
                © 2024 Smartphone Recommendations, All Rights Reserved.
            
        
    
    
    """
     

## Loading web content
This is the complete where the tavily loading all the content available on the web for that particular search

In [12]:
def load_blog_content(page_url):
    try:
        # initialize webBaseLoader with the url
        loader = WebBaseLoader(web_paths=[page_url], bs_get_text_kwargs={"separator": " ", "strip": True})
        loaded_content = loader.load()


        # extract full text from loaded content
        blog_content = " ".join([doc.page_content for doc in loaded_content])

        return blog_content
    
    except Exception as e:
        print(f"Failed to load blog content: {e}")
        return None